# 07 — Covariates and jurisdictional classification

**Input:** `Results/06.xlsx`, `Dictionary/missing_data.json`
**Output:** `Results/07.xlsx`

Consolidates the technical attributes into one value per investment by carrying
forward the most recent reported entry, and derives the model covariates:
infrastructure type, element category, onshore/offshore environment,
jurisdiction, and region.

An investment counts as EU only if all its connection nodes lie within the
EU-27; a link between a Member State and a third country is classified as
Extra-EU, which isolates the effect of the internal regulatory framework from
the slower dynamics of external borders.

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_excel("Results/06.xlsx", header=[0,1], index_col=0)

### Unique Project_ID where possible

In [3]:

years = ['2014','2016', '2018', '2020', '2022', '2024', '2026']

cols_project_id = [(y, 'Project_ID') for y in years if (y, 'Project_ID') in df.columns]
df_project_ids = df.loc[:, cols_project_id].copy()

def unify_project_id(row):
    vals = row.dropna().astype(str).str.strip().unique()
    if len(vals) == 1:
        return vals[0]
    else:
        return np.nan

df[('meta', 'Project_ID')] = df_project_ids.apply(unify_project_id, axis=1)

In [4]:
years = ['2010', '2012', '2013', '2014', '2015', '2016', '2018', '2020', '2022', '2024', '2026']

columns_to_check = [
    'Inv_Capacity [MW]', 
    'Inv_Line length [km]', 
    'Inv_Technology[AC/DC]', 
    'Inv_Voltage [kV]',
    'Inv_type [New-Upgrade]',
    'Project_type [New-Upgrade]',
    'Inv_Element type',
    'Project_Is_cross_border',
]

for colname in columns_to_check:
    cols_present = [(y, colname) for y in years if (y, colname) in df.columns]
    
    if not cols_present:
        continue

    df_subset = df.loc[:, cols_present]
    last_vals = df_subset.ffill(axis=1).iloc[:, -1]
    
    df[('meta', colname)] = last_vals

### Unique jurisdictional analysis (cross border/no)

In [5]:
def count_countries(iso_string):
    """Counts countries based on semicolon-separated ISO codes."""
    if pd.isna(iso_string) or str(iso_string).strip() == "":
        return np.nan
    return len(str(iso_string).split(';'))

df[('meta', 'Project_N_Countries')] = df[('meta', 'Project_Country_ISO3')].apply(count_countries)

def define_crossborder(row):
    """
    Determines if a project is 'Cross-border' or 'Internal'.
    Priority: 1. Existing flag, 2. Deduction from country count.
    """
    cb_flag = row[('meta', 'Project_Is_cross_border')]
    
    if pd.notna(cb_flag):
        val = str(cb_flag).strip().lower()
        if val in ['1', '1.0', 'true', 'cross-border', 'yes']:
            return "Cross-border"
        if val in ['0', '0.0', 'false', 'internal', 'no']:
            return "Internal"
    
    n_countries = row[('meta', 'Project_N_Countries')]
    if pd.notna(n_countries):
        return "Cross-border" if n_countries > 1 else "Internal"
    
    return np.nan

df[('meta', 'Project_Jurisdiction')] = df.apply(define_crossborder, axis=1)

# Check results
print("Country Count Distribution:")
print(df[('meta', 'Project_N_Countries')].value_counts(dropna=False))

print("\nHarmonized Cross-Border Distribution (Labels):")
print(df[('meta', 'Project_Jurisdiction')].value_counts(dropna=False))

df = df.drop(columns=[('meta', 'Project_Is_cross_border')])


Country Count Distribution:
(meta, Project_N_Countries)
1.0    450
2.0    381
3.0     56
4.0     48
NaN     22
8.0      5
5.0      2
6.0      1
Name: count, dtype: int64

Harmonized Cross-Border Distribution (Labels):
(meta, Project_Jurisdiction)
Internal        525
Cross-border    429
NaN              11
Name: count, dtype: int64


In [6]:
# --- EU vs EXTRA-EU ---

# 27 EU Countries
eu_countries = {
    'AUT', 'BEL', 'BGR', 'CYP', 'CZE', 'DEU', 'DNK', 'EST', 'ESP', 
    'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LUX', 
    'LVA', 'MLT', 'NLD', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE'
}

def check_eu_status(country_string):
    if pd.isna(country_string) or str(country_string).strip() == "" or str(country_string).lower() == 'nan':
        return "Unknown"
    
    countries = [c.strip() for c in str(country_string).split(';') if c.strip()]
    
    if not countries:
        return "Unknown"
    
    if all(c in eu_countries for c in countries):
        return "EU"
    else:
        return "Extra-EU"

df[('meta', 'Project_Region')] = df[('meta', 'Project_Country_ISO3')].apply(check_eu_status)

region_counts = df[('meta', 'Project_Region')].value_counts(dropna=False)

print("\n" + "="*30)
print("PROJECT_REGION")
print("="*30)
print(region_counts)
print("-" * 30)
print(f"Total analyzed investments: {region_counts.sum()}")
print("="*30 + "\n")



PROJECT_REGION
(meta, Project_Region)
EU          605
Extra-EU    338
Unknown      22
Name: count, dtype: int64
------------------------------
Total analyzed investments: 965



# New vs Upgrade

In [7]:
df[('meta', 'Infr_Type')] = df[('meta', 'Inv_type [New-Upgrade]')].fillna(df[('meta', 'Project_type [New-Upgrade]')])

df[('meta', 'Infr_Type')] = df[('meta', 'Infr_Type')].str.strip().str.title()

mask_new = df[('meta', 'Infr_Type')].str.contains('New', case=False, na=False)
mask_upgrade = df[('meta', 'Infr_Type')].str.contains('Upgrade', case=False, na=False)

df.loc[mask_new, ('meta', 'Infr_Type')] = 'New'
df.loc[mask_upgrade, ('meta', 'Infr_Type')] = 'Upgrade'

print("Cleaned Infrastructure Type Distribution:")
print(df[('meta', 'Infr_Type')].value_counts(dropna=False))

df = df.drop(columns=[('meta', 'Inv_type [New-Upgrade]'), ('meta', 'Project_type [New-Upgrade]')])

Cleaned Infrastructure Type Distribution:
(meta, Infr_Type)
New             630
NaN             212
Upgrade         121
Missing Data      2
Name: count, dtype: int64


# Infrastructure type

In [8]:
unique_elements = df[('meta', 'Inv_Element type')].unique()

In [9]:
df = df.copy()

def identify_environment(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val)
    # Using 'Offshore' or 'Subsea' to identify maritime projects
    if any(word in val_str for word in ['Subsea', 'Offshore']):
        return 'Offshore'
    return 'Onshore'

df.loc[:, ('meta', 'Inv_Environment')] = df[('meta', 'Inv_Element type')].apply(identify_environment)

def standardize_element(val):
    if pd.isna(val): return np.nan
    s = str(val).lower()
    if any(kw in s for kw in ['line', 'cable']): return 'Cable/Line'
    if any(kw in s for kw in ['substation', 'station', 'transformer', 'pst', 'compensation', 'grid support', 'offshore hub']): 
        return 'Nodal Infrastructure'
    return 'Other'

df[('meta', 'Inv_Element_Category')] = df[('meta', 'Inv_Element type')].apply(standardize_element)


In [10]:
# Define the logical order of columns grouped by category
meta_logical_order = [
    # 1. Identifiers
    'Inv_index',
    'Project_ID',
    
    # 5. Timeline & Report Presence
    'Inv_First_year_present',
    'Inv_Last_year_present',
    'Inv_Years_present',
    
    # 6. Commissioning & Performance
    'Inv_first_reported_commissioning_year',
    'Inv_last_reported_commissioning_year',
    'duration',
    'delay',

    # 7. Final Conclusion
    'Investment Conclusion',
    
    # 2. Geography & Scope
    'Project_Country_ISO3',
    'Project_N_Countries',
    'Project_Jurisdiction',
    'Project_Region',
    
    
    # 3. Technical Details
    'Infr_Type',
    'Inv_Element_Category',
    'Inv_Environment',
    'Inv_Element type',
    'Inv_Technology[AC/DC]',
    'Inv_Capacity [MW]',
    'Inv_Voltage [kV]',
    'Inv_Line length [km]',
    
    # 4. Location (Substations and Coordinates)
    'Inv_Substation_From_Final',
    'Inv_Substation_To_Final',
    'Inv_From_Lat',
    'Inv_From_Lon',
    'Inv_To_Lat',
    'Inv_To_Lon',
    
]

year_cols = [c for c in df.columns if c[0] != 'meta']

ordered_meta_cols = [('meta', col) for col in meta_logical_order if ('meta', col) in df.columns]

df = df[year_cols + ordered_meta_cols]

In [11]:
important_cols = [
    'Project_Jurisdiction', # Internal / Cross-border
    'Project_Region', # EU/Extra-EU
    'Infr_Type', #Upgrade/New
    'Inv_Element_Category', #'Cable/Line' /'Substation'/'Grid Support Equipment'
    'Inv_Environment', #Offshore / Onshore
    'Inv_Technology[AC/DC]',
    'Inv_Capacity [MW]',
    'Inv_Voltage [kV]',
    'Inv_Line length [km]'
]

existing_meta_cols = [('meta', col) for col in important_cols if ('meta', col) in df.columns]

nan_summary = df[existing_meta_cols].isna().sum()
nan_percentage = (df[existing_meta_cols].isna().mean() * 100).round(2)

data_quality_report = pd.DataFrame({
    'Missing Values (Count)': nan_summary,
    'Missing Values (%)': nan_percentage
})

print("--- Data Quality Report: Important Meta Columns ---")
print(data_quality_report.sort_values(by='Missing Values (Count)', ascending=False))

print(f"\nTotal number of rows in dataset: {len(df)}")

--- Data Quality Report: Important Meta Columns ---
                            Missing Values (Count)  Missing Values (%)
Year Column                                                           
meta Inv_Capacity [MW]                         326               33.78
     Inv_Line length [km]                      294               30.47
     Inv_Voltage [kV]                          250               25.91
     Inv_Technology[AC/DC]                     231               23.94
     Infr_Type                                 212               21.97
     Inv_Element_Category                      209               21.66
     Inv_Environment                           209               21.66
     Project_Jurisdiction                       11                1.14
     Project_Region                              0                0.00

Total number of rows in dataset: 965


### Adding new information

In [12]:
file_path = 'Dictionary/missing_data.json'

with open(file_path, 'r') as json_file:
    missing_data = json.load(json_file)

In [13]:
for inv_index_str, updates in missing_data.items():
    inv_index = int(inv_index_str)
    

    mask = df[('meta', 'Inv_index')] == inv_index
    
    if mask.any():
        for column_name, value in updates.items():
            target_col = ('meta', column_name)
            
            if target_col in df.columns:
                df.loc[mask, target_col] = value
            else:
                print(f"Column {column_name} not found under 'meta' level for index {inv_index}")
    else:
        print(f"Investment index {inv_index} not found in the DataFrame.")



Investment index 2045 not found in the DataFrame.


In [14]:
df[('meta', 'Inv_Element_Category')] = df[('meta', 'Inv_Element type')].apply(standardize_element)

In [15]:
existing_meta_cols = [('meta', col) for col in important_cols if ('meta', col) in df.columns]

nan_summary = df[existing_meta_cols].isna().sum()
nan_percentage = (df[existing_meta_cols].isna().mean() * 100).round(2)

data_quality_report = pd.DataFrame({
    'Missing Values (Count)': nan_summary,
    'Missing Values (%)': nan_percentage
})

print(data_quality_report.sort_values(by='Missing Values (Count)', ascending=False))

print(f"\nTotal number of rows in dataset: {len(df)}")

                            Missing Values (Count)  Missing Values (%)
Year Column                                                           
meta Inv_Line length [km]                      263               27.25
     Inv_Capacity [MW]                         247               25.60
     Inv_Voltage [kV]                          136               14.09
     Inv_Technology[AC/DC]                      77                7.98
     Infr_Type                                  56                5.80
     Inv_Element_Category                       56                5.80
     Inv_Environment                            56                5.80
     Project_Jurisdiction                        0                0.00
     Project_Region                              0                0.00

Total number of rows in dataset: 965


In [16]:
years = ['2010', '2012', '2013', '2014', '2015', '2016', '2018', '2020', '2022', '2024', '2026']
cols_to_process = ['Inv_Description', 'Project_Name']

for col_name in cols_to_process:
    existing_year_cols = [(y, col_name) for y in years if (y, col_name) in df.columns]

    if existing_year_cols:
        latest_values = (
            df[existing_year_cols]
            .replace(0, np.nan)           # Treat literal 0 as missing
            .replace('', np.nan)          # Treat empty strings as missing
            .ffill(axis=1)                # Carry forward the last valid entry
            .iloc[:, -1]                  # Take only the last column (the most recent year)
        )

        df[('meta', col_name)] = latest_values

In [17]:

df[('meta', 'Inv_Element_Category')] = df[('meta', 'Inv_Element type')].apply(standardize_element)

In [18]:
df.to_excel("Results/07.xlsx")